# Cloud-Free Sentinel-2 + OSM Data Downloader for Google Colab

This notebook demonstrates how to download:
1. **Cloud-free Sentinel-2 satellite images** from Google Earth Engine
2. **OpenStreetMap (OSM) data** for the same geographic areas
3. **Land-only areas** (filtering out bare ocean)

Perfect for creating custom RSVQA (Remote Sensing Visual Question Answering) datasets!

## 📦 Step 1: Install Required Packages

In [ ]:
!pip install earthengine-api geemap --quiet
print("✓ Packages installed successfully!")

## 🔐 Step 2: Authenticate with Google Earth Engine

**First time only:** You'll need to authenticate and authorize access to Google Earth Engine.

In [ ]:
import ee

# Authenticate (opens browser for authorization)
try:
    ee.Initialize()
    print("✓ Already authenticated!")
except:
    print("Authenticating... Please follow the link and authorize.")
    ee.Authenticate()
    ee.Initialize()
    print("✓ Authentication successful!")

## 📥 Step 3: Upload the Downloader Script

Upload `colab_gee_osm_downloader.py` to your Colab session, or clone the repository:

In [ ]:
# Option A: Clone the repository
!git clone https://github.com/yourusername/RSVQA_learn_v1.git
%cd RSVQA_learn_v1

# Option B: Upload manually (uncomment if not using git)
# from google.colab import files
# uploaded = files.upload()  # Upload colab_gee_osm_downloader.py

## 🚀 Step 4: Import the Downloader

In [ ]:
from colab_gee_osm_downloader import CoLabGEEOSMDownloader
import json
from IPython.display import Image, display

# Initialize the downloader
downloader = CoLabGEEOSMDownloader(output_dir="./downloads")
print("✓ Downloader initialized!")

## 🌍 Step 5: View Predefined Locations

The downloader includes 8 predefined land-based locations worldwide:

In [ ]:
locations = CoLabGEEOSMDownloader.get_predefined_locations()

print("Available predefined locations:\n")
for i, loc in enumerate(locations, 1):
    print(f"{i}. {loc['name']}")
    print(f"   Description: {loc['description']}")
    print(f"   Bbox: {loc['bbox']}")
    print(f"   Cloud threshold: {loc['cloud_percentage']}%\n")

## 📍 Example 1: Download Manhattan, NYC

In [ ]:
# Download Manhattan with cloud-free Sentinel-2 + OSM data
result = downloader.process_location(
    name='manhattan_test',
    bbox=[-74.02, 40.75, -73.97, 40.80],  # [min_lon, min_lat, max_lon, max_lat]
    cloud_percentage=10,
    scale=10  # 10m resolution
)

print("\n" + "="*50)
print("RESULT:")
print(json.dumps(result, indent=2))

## 🌾 Example 2: Download Iowa Farmland

In [ ]:
result = downloader.process_location(
    name='iowa_farmland',
    bbox=[-93.65, 41.55, -93.55, 41.65],
    cloud_percentage=10,
    scale=10
)

print("\n" + "="*50)
print("RESULT:")
print(json.dumps(result, indent=2))

## 🌳 Example 3: Download Amazon Rainforest

In [ ]:
result = downloader.process_location(
    name='amazon_rainforest',
    bbox=[-60.1, -3.1, -60.0, -3.0],
    cloud_percentage=20,  # Higher threshold for rainforest (more clouds)
    scale=10
)

print("\n" + "="*50)
print("RESULT:")
print(json.dumps(result, indent=2))

## 🔄 Example 4: Batch Download Multiple Locations

In [ ]:
import time

# Define custom locations
my_locations = [
    {
        'name': 'paris_france',
        'bbox': [2.30, 48.85, 2.37, 48.88],
        'cloud_percentage': 10
    },
    {
        'name': 'cairo_egypt',
        'bbox': [31.20, 30.02, 31.30, 30.10],
        'cloud_percentage': 5
    },
    {
        'name': 'sydney_australia',
        'bbox': [151.15, -33.92, 151.25, -33.85],
        'cloud_percentage': 10
    }
]

results = []
for loc in my_locations:
    print(f"\nProcessing {loc['name']}...")
    result = downloader.process_location(
        name=loc['name'],
        bbox=loc['bbox'],
        cloud_percentage=loc['cloud_percentage']
    )
    results.append(result)
    
    # Rate limiting to avoid API throttling
    time.sleep(3)

# Summary
print("\n" + "="*50)
print("BATCH PROCESSING SUMMARY")
print("="*50)
successful = [r for r in results if r['status'] == 'success']
print(f"Total: {len(results)} | Success: {len(successful)} | Failed: {len(results) - len(successful)}")

## 📊 Step 6: Inspect Downloaded Data

Let's examine what was downloaded:

In [ ]:
# List all downloaded files
!echo "=== Downloaded Images ==="
!ls -lh downloads/images/

!echo "\n=== OSM Data Files ==="
!ls -lh downloads/osm_data/

!echo "\n=== Metadata Files ==="
!ls -lh downloads/metadata/

## 🔍 Step 7: Examine Metadata

In [ ]:
# Read and display metadata for Manhattan
import json

with open('downloads/metadata/manhattan_test_metadata.json', 'r') as f:
    metadata = json.load(f)

print("Manhattan Metadata:")
print(json.dumps(metadata, indent=2))

## 🗺️ Step 8: Visualize OSM Data

In [ ]:
# Load and summarize OSM data
with open('downloads/osm_data/manhattan_test_osm.geojson', 'r') as f:
    osm_data = json.load(f)

print(f"Total OSM features: {len(osm_data['features'])}")

# Count features by type
from collections import Counter

feature_types = []
for feature in osm_data['features']:
    props = feature['properties']
    # Extract main feature types
    for key in ['building', 'highway', 'landuse', 'natural', 'amenity']:
        if key in props:
            feature_types.append(f"{key}={props[key]}")
            break

counts = Counter(feature_types)
print("\nTop 10 feature types:")
for feature, count in counts.most_common(10):
    print(f"  {feature}: {count}")

## 🖼️ Step 9: Visualize the Satellite Image

In [ ]:
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np

# Install rasterio if not available
!pip install rasterio matplotlib --quiet

# Open and visualize the GeoTIFF
with rasterio.open('downloads/images/manhattan_test.tif') as src:
    # Read RGB bands
    rgb = src.read([1, 2, 3])  # Read bands 1, 2, 3
    
    # Transpose to (height, width, channels) for matplotlib
    rgb = np.transpose(rgb, (1, 2, 0))
    
    # Normalize for visualization
    rgb_norm = np.clip(rgb / 3000 * 255, 0, 255).astype(np.uint8)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(rgb_norm)
    ax.set_title('Manhattan - Cloud-Free Sentinel-2 Composite', fontsize=16)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Image shape: {rgb.shape}")
    print(f"CRS: {src.crs}")
    print(f"Bounds: {src.bounds}")

## 📥 Step 10: Download Results to Your Computer

In [ ]:
# Zip all downloads for easy download
!zip -r downloads.zip downloads/

# Download the zip file
from google.colab import files
files.download('downloads.zip')

print("✓ Downloads ready! Check your browser's download folder.")

## 🎯 Custom Location Example

Download data for your own custom location:

In [ ]:
# Define your custom location
# You can use https://boundingbox.klokantech.com/ to get coordinates

my_custom_location = {
    'name': 'my_study_area',
    'bbox': [-122.5, 37.7, -122.4, 37.8],  # Replace with your coordinates
    'cloud_percentage': 10
}

result = downloader.process_location(
    name=my_custom_location['name'],
    bbox=my_custom_location['bbox'],
    cloud_percentage=my_custom_location['cloud_percentage'],
    scale=10
)

print(json.dumps(result, indent=2))

## 🔧 Advanced: Custom OSM Features

Specify exactly which OSM features to download:

In [ ]:
# Download only specific OSM features
result = downloader.process_location(
    name='manhattan_buildings_only',
    bbox=[-74.02, 40.75, -73.97, 40.80],
    cloud_percentage=10,
    osm_features=['building', 'highway', 'amenity']  # Only these features
)

print(json.dumps(result, indent=2))

## 📚 Next Steps

Now that you have cloud-free Sentinel-2 images with OSM data, you can:

1. **Tile images** into 256×256 patches using `prepare_sentinel2_data.py`
2. **Generate questions** from OSM features using `annotation_helper.py`
3. **Create dataset splits** with `create_dataset_splits.py`
4. **Train VQA models** using the RSVQA framework

See the repository's documentation for complete workflows:
- `PHASE1_DOWNLOAD_GUIDE.md`
- `CUSTOM_DATASET_WORKFLOW.md`
- `ANNOTATION_GUIDE.md`

## 🐛 Troubleshooting

### GEE Authentication Issues
```python
# Re-authenticate if needed
import ee
ee.Authenticate(force=True)
ee.Initialize()
```

### No Images Found
- Try increasing `cloud_percentage` threshold (15-20%)
- Expand the date range (last 2 years)
- Check if the area has Sentinel-2 coverage

### OSM API Timeout
- Reduce the bounding box size
- Specify fewer OSM features
- Wait a few minutes and retry (rate limiting)

### Land Detection Issues
- The script checks `mean_brightness` and `std_dev`
- Coastal areas may trigger warnings (expected)
- Pure ocean areas will be flagged but still downloaded